# 000. Data Pipeline
- This gathers single-run daily data and extracts updated data from yesterday 
- Type: Pipeline
- Run Frequency: Daily (Morning)
- Created: 1/1/2025
- Updated: 5/29/2025

In [1]:
from DataImports import *

### Settings

In [2]:
# hard code for testing
yesterdaysdate = '20250101'
todaysdate = '20251231'

start_date, end_date = yesterdaysdate, todaysdate

### Games

In [3]:
all_game_df = pd.read_csv(os.path.join(baseball_path, "game_df.csv"))

In [4]:
game_df = all_game_df[(all_game_df['date'].astype(str) >= start_date) & (all_game_df['date'].astype(str) <= end_date)].reset_index(drop=True)

In [5]:
game_df = pd.merge(game_df, venue_map_df[['id', 'location.defaultCoordinates.latitude', 'location.defaultCoordinates.longitude',
                                          'fieldInfo.leftLine', 'fieldInfo.center', 'fieldInfo.rightLine', 'fieldInfo.leftCenter', 'fieldInfo.rightCenter', 
                                          'location.elevation', 'location.azimuthAngle', 'fieldInfo.roofType', 'active']], 
                                           left_on=['venue_id'], right_on=['id'], how='left')

In [6]:
game_df["game_datetime"] = pd.to_datetime(game_df["game_datetime"])

In [7]:
game_df[['home_probable_pitcher', 'away_probable_pitcher']] = game_df[['home_probable_pitcher', 'away_probable_pitcher']].fillna("Missing Starter")

### A01. Player Results

Date(s): Yesterday

In [ ]:
df = game_df[game_df['date'].astype(str) == yesterdaysdate].reset_index(drop=True)

In [ ]:
# _ = Parallel(n_jobs=-1)(delayed(run_player_results)(df, row) for row in range(len(df)))

### A02. MLB API

Date(s): Year-to-Date

In [ ]:
year = int(start_date[:4])

##### 1. Stats API

In [ ]:
# statsapi_df = plays_statsapi(f"03/18/{year}", f"11/15/{year}")
# statsapi_df.to_csv(os.path.join(baseball_path, "A02. MLB API", "1. Stats API", f"Stats API {year}.csv"), index=False, encoding='iso-8859-1')

##### 2. Statcast

In [ ]:
# statcast_df = plays_statcast(f"{year}-03-18", yesterdaysdate_dash)
# statcast_df.to_csv(os.path.join(baseball_path, "A02. MLB API", "2. Statcast", f"Statcast {year}.csv"), index=False, encoding='iso-8859-1')

### A03. FanGraphs

Date(s): Daily

##### Steamer

In [ ]:
# fangraphs_api("https://www.fangraphs.com/api/projections?type=steamer&stats=bat&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "Steamer", f"Steamer Hitters {todaysdate}.csv"), index=False)
# fangraphs_api("https://www.fangraphs.com/api/projections?type=steamer&stats=pit&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "Steamer", f"Steamer Pitchers {todaysdate}.csv"), index=False)

##### ZiPS

In [ ]:
# fangraphs_api("https://www.fangraphs.com/api/projections?type=zips&stats=bat&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "ZiPS", f"ZiPS Hitters {todaysdate}.csv"), index=False)
# fangraphs_api("https://www.fangraphs.com/api/projections?type=zips&stats=pit&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "ZiPS", f"ZiPS Pitchers {todaysdate}.csv"), index=False)

##### THE BAT X

In [ ]:
# fangraphs_api("https://www.fangraphs.com/api/projections?type=thebatx&stats=bat&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "THE BAT X", f"THE BAT X Hitters {todaysdate}.csv"), index=False)
# fangraphs_api("https://www.fangraphs.com/api/projections?type=thebatx&stats=pit&pos=all&team=0&players=0&lg=all").to_csv(os.path.join(baseball_path, "A03. FanGraphs", "THE BAT X", f"THE BAT X Pitchers {todaysdate}.csv"), index=False)

### A04. Bullpens

Date(s): Today

In [ ]:
historic = False

In [ ]:
# _ = Parallel(n_jobs=-1, verbose=0)(delayed(bullpens)(date=date, team_map=team_map, historic=historic) for date in [todaysdate])

### A05. Rosters

##### 1. Batting Orders

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(str) >= yesterdaysdate) & (game_df['date'].astype(str) <= todaysdate)].reset_index(drop=True)

In [ ]:
# _ = Parallel(n_jobs=-1, verbose=0)(delayed(orders)(team_map, df, row) for row in range(len(df)))

##### 2. Rosters

Date(s): Yesterday and Today

In [ ]:
# _ = Parallel(n_jobs=-1, verbose=0)(delayed(rosters)(team_map, df, row) for row in range(len(df)))

##### 3. Projected Lineups - RotoGrinders

Date(s): Today

In [ ]:
# rotogrinders_lineups_df = scrape_rotogrinders_lineups()
# rotogrinders_lineups_df.to_csv(os.path.join(baseball_path, "A05. Rosters", "3. Projected Lineups - RotoGrinders", f"{todaysdate} Projected Lineups - RotoGrinders.csv"), index=False)

 ##### 4. Projected Lineups - Baseball Monster

Date(s): Today

In [ ]:
# projected_batting_orders = pd.read_csv(rf"https://baseballmonster.com/Lineups.aspx?csv=1&d={todaysdate_dash}")
# projected_batting_orders.to_csv(os.path.join(baseball_path, "A05. Rosters", "4. Projected Lineups - Baseball Monster", f"{todaysdate} Projected Lineups - Baseball Monster.csv"), index=False)

### A06. Weather

##### 1. Open Meteo

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(str) >= yesterdaysdate) & (game_df['date'].astype(str) <= todaysdate)].reset_index(drop=True)

Setup the Open-Meteo API client **without** cache and with retry on error

In [ ]:
# base_session = requests.Session()
# retry_session = retry(base_session, retries=5, backoff_factor=0.2)
# openmeteo = openmeteo_requests.Client(session=retry_session)

In [ ]:
# for date in df['date'].unique():
#     print(date)
#     if int(date) == int(todaysdate):
#         print("Open Meteo: Today")
#         open_meteo_df = create_daily_weather_df(openmeteo, df[df['date'].astype(int) == date])[game_columns + venue_columns + weather_columns + forecast_only_columns]
#     else:
#         print("Open Meteo: Historic")
#         open_meteo_df = create_historic_weather_df(openmeteo, df[df['date'] == date])[game_columns + venue_columns + weather_columns]

#     open_meteo_df.to_csv(os.path.join(baseball_path, "A06. Weather", "1. Open Meteo", f"Open Meteo {date}.csv"), index=False)
#     time.sleep(2)

##### 2. RotoGrinders

Date(s): Today

In [ ]:
# try:
#     rotogrinders_df = rotogrinders_weather(todaysdate, team_map)
#     rotogrinders_df.to_csv(os.path.join(baseball_path, "A06. Weather", "2. RotoGrinders", f"RotoGrinders {todaysdate}.csv"), index=False)
# except:
#     print("Could not scrape RotoGrinders weather data.")

### A07. Odds

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(str) >= yesterdaysdate) & (game_df['date'].astype(str) <= todaysdate)].reset_index(drop=True)

In [ ]:
# # %%
# for date_dash in df['game_date'].unique():
#     try:
#         # Scrape odds from Sportsbook Review
#         odds_df = create_odds_df(date_dash)

#         date = date_dash.replace("-", "")

#         # Save 
#         odds_df.to_csv(os.path.join(baseball_path, "A07. Odds Sportsbook Review", f"Odds {date}.csv"), index=False) 

#         # Sleep to avoid hitting rate limit
#         time.sleep(1)

#     except:
#         print(f"Sportsbook Review data unavailable for {date_dash}")

### A08. Projections

Date(s): Yesterday and Today

##### 1. DFF

1. Slates

In [ ]:
# dff_slates_df = dff_slates(todaysdate)
# dff_slates_df.to_csv(os.path.join(baseball_path, "A08. Projections", "1. DFF", "1. Slates", f"DFF Slates {todaysdate}.csv"), index=False)

2. Projections

In [ ]:
# for code in dff_slates_df['URL']:
#     try:
#         # Scrape projections
#         dff_projections_df = dff_projections(todaysdate, code)
#         # To CSV
#         dff_projections_df.to_csv(os.path.join(baseball_path, "A08. Projections", "1. DFF", "2. Projections", f"DFF Projections {code}.csv"), index=False)
#     except KeyError as e:
#         print("KeyError")

##### 2. RotoWire

1. Slates

In [ ]:
# roto_slates_df = roto_slates(todaysdate)
# roto_slates_df.to_csv(os.path.join(baseball_path, "A08. Projections", "2. RotoWire", "1. Slates", f"RotoWire Slates {todaysdate}.csv"), index=False)

2. Projections

In [ ]:
# try:
#     for code in roto_slates_df['slateID']:
#         # Scrape projections
#         roto_projections_df = roto_projections(todaysdate, code)
#         try:
#             # To CSV
#             roto_projections_df.to_csv(os.path.join(baseball_path, "A08. Projections", "2. RotoWire", "2. Projections", f"RotoWire Projections {code}.csv"), index=False)
#         except:
#             print(f"Can't scrape RotoWire projections for slate {code}.")
# except:
#     print("RotoWire projections unavailable.")

### A09. DraftKings

Date(s): Yesterday and Today

##### 1. Contests

In [ ]:
# # Scrape contests
# contest_df = contests(todaysdate)
# # Write to CSV
# contest_df.to_csv(os.path.join(baseball_path, 'A09. DraftKings', '1. Contests', f'Contests {todaysdate}.csv'), index=False)

##### 7. Subsets

In [ ]:
# # Select subset of contests
# subset_df = create_subset(contest_df, contests_per_draftGroupId=5, entry_fee_max=100, added_contestKeys=[], date_dash=todaysdate_dash)
# # Write to CSV
# subset_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "7. Subsets", f'Subset {todaysdate}.csv'), index=False)

##### 2. Draftables

In [ ]:
# # Loop over unique slates in subset of contests
# for draftGroupId in list(subset_df['draftGroupId'].unique()):
#     try:
#         # Scrape draftables (Salaries)
#         draftable_df = draftables(draftGroupId)
#         # Write to CSV
#         draftable_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "2. Draftables", f"Draftables {draftGroupId}.csv"), index=False, encoding='iso-8859-1')
#         print(f"Saved {draftGroupId}")
#     except:
#         print(f"Didn't save {draftGroupId}")

##### 3. Payouts

In [ ]:
# # Loop over contests of interest
# for i in range(len(subset_df)):
#     # Extract contestKey
#     contestKey = subset_df['contestKey'][i]
#     try:
#         # Scrape payouts
#         payout_df = payouts(contestKey)
#         # Write to CSV
#         payout_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "3. Payouts", f"Payouts {contestKey}.csv"), index=False, encoding='iso-8859-1')
#     except:
#         print(f"Didn't save {contestKey}.")


##### 4. Results, 5. Entry Results, and 6. Player Results

In [ ]:
# try:
#     yesterdays_subset_df = pd.read_csv(os.path.join(baseball_path, 'A09. DraftKings', '7. Subsets', f'Subset {yesterdaysdate}.csv'))

#     # Loop over yesterday's contests
#     for i in range(len(yesterdays_subset_df)):
#         # Extract contestKey
#         contestKey = yesterdays_subset_df['contestKey'][i]
#         print(contestKey)
#         # 4. Results
#         # Download
#         results(contestKey, sleep_time=5)

#         try:
#             # Read into pandas, if possible
#             results_df = pd.read_csv(os.path.join(baseball_path, "A09. DraftKings", "4. Results", f"contest-standings-{contestKey}.csv"))

#             # 5. Entry Results
#             # Extract contest results
#             entry_results_df = entry_results(results_df)
#             # Write to CSV
#             entry_results_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "5. Entry Results", f"Entry Results {contestKey}.csv"), index=False, encoding='iso-8859-1')

#             # 6. Player Results
#             # Extract player results
#             player_results_df = player_results(results_df)
#             # Write to CSV
#             player_results_df.to_csv(os.path.join(baseball_path, "A09. DraftKings", "6. Player Results", f"Player Results {contestKey}.csv"), index=False, encoding='iso-8859-1')
#         except IndexError as e:
#             print(f"Downloaded contest-standings-{contestKey}. Non-traditional format.")
#         except pd.errors.EmptyDataError as e:
#             print(f"Downloaded contest-standings-{contestKey}. Empty file.")
#         except FileNotFoundError as e:
#             print(f"Couldn't find {contestKey}.")
# except FileNotFoundError as e:
#     print(f"Couldn't find yesterday's subset file.")   

### M01. Park and Weather Factors

Date(s): All

In [ ]:
# %run "M01. Park and Weather Factors.ipynb"

### B01. Matchups

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(int) >= int(yesterdaysdate)) & (game_df['date'].astype(int) <= int(todaysdate))].reset_index(drop=True)

In [ ]:
# create_all_matchups(game_df=df, baseball_path=baseball_path, team_dict=team_dict, year=year, todaysdate_dash=todaysdate_dash, batter_inputs=batter_inputs, pitcher_inputs=pitcher_inputs, n_jobs=16)

### B02. WFX

Date(s): Today <br>
Note: Yesterday will be fixed in M01

In [ ]:
# wfx_df = create_wfx_df(similar_games=50, date=todaysdate)
# wfx_df.to_csv(os.path.join(baseball_path, "B02. Weather", f"Park and Weather Factors {todaysdate}.csv"), index=False, encoding='iso-8859-1')

### B03. Contest Guides

Date(s): Yesterday and Today

In [ ]:
df = game_df[(game_df['date'].astype(str) >= yesterdaysdate) & (game_df['date'].astype(str) <= todaysdate)].reset_index(drop=True)

In [ ]:
# for date in df['date'].unique():
#     print(date)
#     # Read in contest subset
#     subset_df = pd.read_csv(os.path.join(baseball_path, 'A09. DraftKings', '7. Subsets', f'Subset {date}.csv'))
    
#     # Loop over contestKeys
#     for contestKey in subset_df['contestKey'].reset_index(drop=True):
#         print(contestKey)
#         try:
#             guide = contest_guide(df, subset_df=subset_df, contestKey=contestKey)
#             if not guide.empty:
#                 guide.to_csv(os.path.join(baseball_path, "B03. Contest Guides", f"Contest Guide {contestKey}.csv"), index=False)
#             else:
#                 print(f"Contest Guide {contestKey} is empty.")
#         except FileNotFoundError as e:
#             print(f"Draftables {contestKey}.csv not found.")